In [1]:
!python --version

Python 3.11.14


## GDC

https://docs.gdc.cancer.gov/API/Users_Guide/Downloading_Files/

sample  
 ├── primary_site       (organ)  
 ├── disease_type       (broad class)  
 ├── subtype_global     (cross-cancer)  
 ├── subtype_tissue     (within-cancer)  
 └── histology          (biological family)  

### TCGA / GDC Portal

https://portal.gdc.cancer.gov/

### 🧠 Big picture (this is powerful)

- You’ve essentially built the backbone of:
- cBioPortal-like explorer
- cohort stratification engine
- patient-specific pipeline (your perturbation_agent 🔥)

### 💡 Flow

> project → project_id  
> primary_sites → pid and disease_type  
> subtypes → subtype_id  
> stages →  stage_id  
> case_id → case_ids or UUIDs  
> samples → sample_id [tumor, normal]  
> aliquots → aliquot_id  
> files → file_id  


In [2]:
import os, sys
import pandas as pd

from pathlib import Path

ROOT = Path().resolve().parent.parent
SRC = os.path.join(ROOT, "src")

if str(SRC) not in sys.path:
    sys.path.append(str(SRC))

print("ROOT:", ROOT)
print("SRC added:", SRC)

from libs.calc_degs_lib import CALC_DEGS
from libs.tcga_gdc_lib import *
from libs.Basic import *


ROOT: /home/flavio/uv
SRC added: /home/flavio/uv/src


### 👉 a TCGA cohort builder + data retrieval engine

Which means you can easily extend to:


> pan-cancer analyses  
> patient-specific pipelines (your perturb_agent 🔥)  
> automated workflows (Nextflow-ready)  

#### 🚀 Next steps

> Production pipeline  
  - CLI tool
  - cached queries
  - reproducible outputs

> Expression matrix builder  
  - auto-download
  - merge TSVs
  - DESeq2-ready matrix

> Patient-level analysis
  - per-case DEG
  - pathway perturbation scoring

### Defaults

In [3]:
ROOT = Path().resolve().parent
root_data = os.path.join(ROOT, "data/tcga")

gdc = GDC(root_data=root_data)

os.listdir(root_data)[:10]


["samples_for_TCGA-CESC_Cervix_uteri_Ovary_subtype_'adenocarcinoma_generic'_tumor_'adenocarcinoma'_subtype_tissue_'adenocarcinoma_generic'.tsv",
 "samples_for_TCGA-LUAD_Bronchus_and_lung_subtype_'adenocarcinoma_generic'_tumor_'adenocarcinoma'_subtype_tissue_'lung_adenocarcinoma'.tsv",
 'cases_for_TCGA-LUSC.tsv',
 "samples_for_TCGA-GBM_Brain_subtype_'adenocarcinoma_generic'_tumor_'adenocarcinoma'_subtype_tissue_'adenocarcinoma_generic'.tsv",
 "samples_for_TCGA-LUAD_Bronchus_and_lung_subtype_'squamous'_tumor_'squamous_cell_carcinoma'_subtype_tissue_'squamous'.tsv",
 "samples_for_TCGA-BRCA_Breast_subtype_'clear_cell'_tumor_'other'_subtype_tissue_'clear_cell'.tsv",
 'cases_for_TCGA-PCPG.tsv',
 "samples_for_TCGA-ACC_Adrenal_gland_subtype_'other'_tumor_'other'_subtype_tissue_'other'.tsv",
 "mutations_anal_for_study_TCGA-ACC_Adrenal_gland_subtype_'adrenal_cortical_carcinoma'_tumor_'adrenal_cortical_carcinoma'_subtype_tissue_'adrenal_cortical_carcinoma'_samples_TCGA-OR-A5J7-01_to_TCGA-OR-A5J7-

### Subtypes explanation

👉 GDC does NOT give you clean biological subtypes
👉 You must derive them yourself

This is actually a feature, not a bug — because:

> you control granularity  
> you can harmonize across cancers  
> you avoid noisy labels  


💡 If you want to level this up

I can help you build:

🔬 A cross-TCGA subtype harmonizer
maps all projects → unified subtype ontology
handles synonyms (adenocarcinoma, NOS, etc.)
outputs clean ML-ready labels

👉 That would massively improve your perturbation analysis.

### Get all programs

In [4]:
force=False
verbose=True

prog_list = gdc.get_gdc_progams(force=force, verbose=verbose)


File read at '/home/flavio/uv/perturb_agent/data/tcga/programs.txt'


In [5]:
np.array(prog_list)

array(['TCGA', 'MATCH', 'TARGET', 'CGCI', 'CMI', 'APOLLO', 'BEATAML1.0',
       'CPTAC', 'MP2PRT', 'ALCHEMIST', 'CCDI', 'CCG', 'CDDP_EAGLE',
       'CTSP', 'EXCEPTIONAL_RESPONDERS', 'FM', 'HCMI', 'MMRF', 'NCICCR',
       'OHSU', 'ORGANOID', 'RC', 'REBC', 'TRIO', 'VAREPOP', 'WCDT'],
      dtype='<U22')

### Primary sites given a program

In [6]:
gdc.url_gdc_project

'https://api.gdc.cancer.gov/projects'

In [ ]:
def loop_program_psi_samples(program:str='TCGA', ipsi:Any=None, force:bool=False, verbose:bool=True) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    df_psi = gdc.get_primary_sites(program=program, force=force, verbose=verbose)

    df_cases, df_subt, df_prof = pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

    if isinstance(ipsi, int):
        lista = [ipsi]
    else:
        lista = np.arange(len(df_psi))

    df_list_cases = []
    df_list_samples = []
    for ipsi in lista:
        row = df_psi.iloc[ipsi]
        pid = row.pid
        primary_site = row.primary_site

        df_cases, df_subt, df_prof = gdc.get_cases_and_subtypes(pid=pid, batch_size=200, do_filter=False, force=force, verbose=verbose)

        if df_cases.empty:
            print("No cases found for PID:", pid)
            continue

        df_list_cases.append(df_cases)

        for isubt, row in df_subt.iterrows():
            subtype_global = row.subtype_global
            tumor_class = row.tumor_class
            subtype_tissue = row.subtype_tissue

            s_case = f"{pid}_{primary_site}_subtype_{subtype_global}_tumor_{tumor_class}_subtype_tissue_{subtype_tissue}"
            s_case = title_replace(s_case)

            if len(s_case) > 200:
                s_case = f"{pid}_{primary_site[:40]}_subtype_{subtype_global[:40]}_tumor_{tumor_class[:40]}_subtype_tissue_{subtype_tissue[:40]}"
                s_case = title_replace(s_case)


            print(f'{isubt}) {s_case}')

            df_samples = gdc.get_samples_for_pid_subtypes(pid=pid, subtype_global=subtype_global,
                                                          tumor_class=tumor_class, subtype_tissue=subtype_tissue, s_case=s_case,
                                                          batch_size=200, force=force, verbose=verbose)
            
            if df_samples.empty:
                print("No samples found for PID:", pid)
                continue


            df2 = df_samples[~df_samples.sample_type.str.contains('Blood', case=False, na=False)]

            if df2.empty:
                print("No samples having non-blood types for PID:", pid)
                continue

            barcodes = list(np.unique(df_samples.barcode_id))

            print("\nGetting mutations", end=' ')

            dff, _ = gdc.get_df_mut_transform_mutation_table(study_id=pid, s_case=s_case, sample_ids=barcodes, force=force, verbose=verbose)

            if dff.empty:
                print("Could not find mutations for :", s_case)
                continue

            df_list_samples.append(dff)

    if len(df_list_cases) > 0:
        df_all_cases = pd.concat(df_list_cases, ignore_index=True)
        df_all_cases = df_all_cases.drop_duplicates()
        df_all_cases = df_all_cases.reset_index(drop=True)
    else:
        df_all_cases = pd.DataFrame()

    if len(df_list_samples) > 0:
        df_all_samples = pd.concat(df_list_samples, ignore_index=True)
        df_all_samples = df_all_samples.drop_duplicates()
        df_all_samples = df_all_samples.reset_index(drop=True)
    else:
        df_all_samples = pd.DataFrame()

    return df_psi, df_all_cases, df_all_samples



In [8]:
force=False
verbose=False

df_psi, df_all_cases, df_all_samples = loop_program_psi_samples(program='TCGA', ipsi=None, force=force, verbose=verbose)

print("\n----------- end ------------\n")
print(len(df_all_samples))


0) TCGA-ACC_Adrenal_gland_subtype_'adrenal_cortical_carcinoma'_tumor_'adrenal_cortical_carcinoma'_subtype_tissue_'adrenal_cortical_carcinoma'

Getting mutations 1) TCGA-ACC_Adrenal_gland_subtype_'adrenal_cortical_carcinoma'_tumor_'adrenal_cortical_carcinoma'_subtype_tissue_'adrenal_cortical_carcinoma'

Getting mutations 2) TCGA-ACC_Adrenal_gland_subtype_'other'_tumor_'other'_subtype_tissue_'other'

Getting mutations 3) TCGA-ACC_Adrenal_gland_subtype_'sarcoma'_tumor_'sarcoma'_subtype_tissue_'sarcoma'

Getting mutations Error: cBioPortal URL: https://www.cbioportal.org/api/molecular-profiles/acc_tcga_mutations/mutations/fetch
No mutations found for molecular profile 'acc_tcga_mutations' samples: ['TCGA-P6-A5OG-01', 'TCGA-P6-A5OG-01', 'TCGA-P6-A5OG-10'].
No mutations found for these samples.
Could not find mutations for : TCGA-ACC_Adrenal_gland_subtype_'sarcoma'_tumor_'sarcoma'_subtype_tissue_'sarcoma'
0) TCGA-PCPG_Adrenal_gland_Retroperitoneum_and_peritoneum_Other_endocrine_glands_and_re

OSError: [Errno 36] File name too long: "/home/flavio/uv/perturb_agent/data/tcga/mutations_summ_for_study_TCGA-PCPG_Adrenal_gland_Retroperitoneum_and_peritoneum_Other_endocrine_glands_and_related_structures_Other_and_ill-defined_sites_Connective_subcutaneous_and_other_soft_tissues_Spinal_cord_cranial_nerves_and_other_parts_of_central_nervous_system_Heart_mediastinum_and_pleura_subtype_'other'_tumor_'other'_subtype_tissue_'other'_samples_TCGA-QR-A6GT-01_to_TCGA-QR-A6GT-01.tsv"

In [10]:
s_case = "TCGA-ACC_Adrenal_gland_subtype_'adrenal_cortical_carcinoma'_tumor_'adrenal_cortical_carcinoma'_subtype_tissue_'adrenal_cortical_carcinoma'"
len(s_case)

138